# bottleneck-latent-projection — ex1: encode flat batch into latent + 2-D scatter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bottleneck-latent-projection`. Running the final beacon cell reports progress against the `Generative: Bottleneck latent projection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Bottleneck latent projection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bottleneck-latent-projection`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bottleneck-latent-projection"
DD_SUBTOPIC = "Generative: Bottleneck latent projection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Bottleneck latent projection — quick refresher

The bottleneck is the SINGLE narrowest layer in an autoencoder. It is almost always a `nn.Linear(flattened_dim, latent_dim)` where `latent_dim << flattened_dim`. Two non-obvious things:

1. **Dim reduction is the entire job.** No activation, no normalization lives at the bottleneck itself — just `Linear`. Any non-linearity goes BEFORE the projection (in the encoder body) or AFTER (in the decoder body), never on the bottleneck output.
2. **Input is already flat.** ARENA flattens with `Rearrange('b c h w -> b (c h w)')` before the bottleneck Linear. The bottleneck does not know about spatial structure — it sees a 1-D feature vector per batch element.

**Why it matters.** Visualizing the latent space (2-D scatter colored by class label) tells you whether the bottleneck has learned a useful compression. A well-trained autoencoder produces class-clustered latents; a random-weights one produces a featureless blob.

### Exercise 1 — encode flat batch into latent + 2-D scatter

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a single `nn.Linear(input_dim, latent_dim)` to project a flat `(B, 784)` batch into a `(B, latent_dim)` latent code, with no non-linearity on the bottleneck output.
> Keywords: bottleneck, Linear, latent-projection, visualization
> ```

**KCs targeted:** `bottleneck-linear-projection`, `bottleneck-dim-reduction`

Implement `ex1_bottleneck_project(flat_batch, weight, bias)`. The ATOMIC bottleneck operation that every autoencoder hides inside its encoder:

1. `flat_batch` has shape `(B, 784)` — already flattened MNIST.
2. `weight` has shape `(latent_dim, 784)` and `bias` has shape `(latent_dim,)` — together they parameterize an `nn.Linear(784, latent_dim)` (PyTorch stores `weight` as `(out_features, in_features)`).
3. Compute `latent = flat_batch @ weight.T + bias`.
4. **DO NOT** apply ReLU, sigmoid, layer-norm, or anything else — the bottleneck output is the bare Linear projection.

Input: `(B, 784)` float tensor.
Output: `(B, latent_dim)` float tensor.

The visualization runs your function on a synthetic 3-class dataset with `latent_dim=2` and scatters the latent codes colored by class — well-chosen weights produce three visible clusters.

In [ ]:
def ex1_bottleneck_project(flat_batch: Tensor, weight: Tensor, bias: Tensor) -> Tensor:
    """Project (B, 784) -> (B, latent_dim) via a single Linear."""
    raise NotImplementedError()


def _test_ex1():
    # Smoke test — known shapes.
    B, in_dim, latent_dim = 4, 784, 5
    flat = t.randn(B, in_dim, generator=t.Generator().manual_seed(0))
    W = t.randn(latent_dim, in_dim, generator=t.Generator().manual_seed(1))
    b = t.randn(latent_dim, generator=t.Generator().manual_seed(2))
    out = ex1_bottleneck_project(flat, W, b)
    assert out.shape == (B, latent_dim), f'expected (B,latent_dim)=({B},{latent_dim}), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'

    # Numerical correctness — must equal the bare affine.
    expected = flat @ W.T + b
    assert t.allclose(out, expected, atol=1e-5), 'output must equal flat @ W.T + b'

    # Identity case — W=I (square), b=0 → should reproduce the input.
    id_W = t.eye(in_dim)
    id_b = t.zeros(in_dim)
    out_id = ex1_bottleneck_project(flat, id_W, id_b)
    assert t.allclose(out_id, flat, atol=1e-5), 'identity projection must be a no-op'

    # Catch: a ReLU on the output would clip negatives — guard against it.
    neg_flat = -t.ones(2, in_dim)
    neg_W = t.eye(in_dim)
    neg_b = t.zeros(in_dim)
    out_neg = ex1_bottleneck_project(neg_flat, neg_W, neg_b)
    assert (out_neg < 0).all(), 'bottleneck has NO ReLU — negatives must pass through'

    # --- Visualization: 2-D latents colored by class ---
    rng = t.Generator().manual_seed(7)
    K = 3                  # classes
    Nc = 60                # samples per class
    in_dim_viz = 784
    latent_dim_viz = 2
    # Each class has a different mean in input space.
    centers = t.randn(K, in_dim_viz, generator=rng)
    labels = t.cat([t.full((Nc,), c, dtype=t.long) for c in range(K)])
    data = t.cat([centers[c].unsqueeze(0) + 0.3 * t.randn(Nc, in_dim_viz, generator=rng) for c in range(K)], dim=0)
    # Weights chosen to separate centers in 2-D.
    W_viz = t.stack([centers[0] - centers[1], centers[0] - centers[2]], dim=0)  # (2, 784)
    b_viz = t.zeros(latent_dim_viz)
    latents = ex1_bottleneck_project(data, W_viz, b_viz)
    fig, ax = plt.subplots(figsize=(5, 5))
    for c in range(K):
        mask = labels == c
        ax.scatter(latents[mask, 0].numpy(), latents[mask, 1].numpy(),
                   label=f'class {c}', alpha=0.7, s=25)
    ax.set_xlabel('latent dim 0')
    ax.set_ylabel('latent dim 1')
    ax.set_title('ex1 bottleneck latents (2-D scatter, colored by class)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_bottleneck_project(flat_batch: Tensor, weight: Tensor, bias: Tensor) -> Tensor:
    return flat_batch @ weight.T + bias
```

**Why `weight.T`.** PyTorch stores `nn.Linear` weights as `(out_features, in_features)` so that `Linear(in, out).weight` has shape `(out, in)` — that's the row-per-output convention used in papers. To apply it to `(B, in)` input you need the transpose so the contraction is `(B, in) @ (in, out) → (B, out)`.

**The 2-D scatter is the diagnostic.** A bottleneck whose latents blob into one cluster has learned nothing. A bottleneck whose latents form K distinct clusters has implicitly learned a class-separating compression — without ever being trained on labels. That's the magic of autoencoders.

**No activation on the bottleneck.** Adding ReLU here would clip negative latents — half the representation space — for no benefit. Adding BatchNorm or LayerNorm would collapse the very dim reduction you're trying to measure.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()